In [68]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd
import json
import re
from tqdm import tqdm
from PIL import Image, ImageOps, ImageDraw, ImageFont

from src.utilities_screen_peak_combined_single_fig import *        # use this instead to not substract mean

import warnings
warnings.filterwarnings("ignore")

protein_ctrl = 'NPC'
compound_ctrl = 'NCC'

In [70]:
def load_images(paths):
    imgs = []
    for p in paths:
        #try:
        im = Image.open(p).convert("RGB")
        imgs.append(im)
        #except Exception as e:
         #   print(f"Skipping {p}: {e}")
    return imgs
    
def save_three_on_one_pdf(images, texts, out_pdf="figures.pdf"):
    PAGE_W, PAGE_H = 2480, 3508  # A4 at 300 DPI
    MARGIN = 120
    ROW_GAP = 80
    CAPTION_H = 80   # space reserved for text
    BG_COLOR = "white"

    ims = [im.convert("RGB") if isinstance(im, Image.Image) else Image.open(im).convert("RGB") for im in images]
    if len(ims) != 3 or len(texts) != 3:
        raise ValueError("Expected exactly 3 images and 3 texts")

    usable_h = PAGE_H - 2*MARGIN - 2*ROW_GAP
    slot_h = usable_h // 3
    slot_w = PAGE_W - 2*MARGIN

    page = Image.new("RGB", (PAGE_W, PAGE_H), BG_COLOR)
    draw = ImageDraw.Draw(page)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Supplemental/Helvetica.ttc", 50)
    except:
        font = ImageFont.load_default()

    for idx, (im, txt) in enumerate(zip(ims, texts)):
        x0 = MARGIN
        y0 = MARGIN + idx * (slot_h + ROW_GAP)

        # --- FIX: use textbbox instead of textsize ---
        bbox = draw.textbbox((0, 0), txt, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        tx = x0 + (slot_w - tw) // 2
        draw.text((tx, y0), txt, fill="black", font=font)

        # Adjust slot so image sits below caption
        y0 += CAPTION_H
        w, h = im.size
        scale = min(slot_w / w, (slot_h - CAPTION_H) / h)
        im_fit = im.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        im_fit = ImageOps.expand(im_fit, border=2, fill="black")
        x = x0 + (slot_w - im_fit.width) // 2
        y = y0 + (slot_h - CAPTION_H - im_fit.height) // 2
        page.paste(im_fit, (x, y))

    page.save(out_pdf)
    # print(f"Saved {out_pdf}")

In [74]:
input_path = '/Users/behnam/Desktop/My_Codes/screen_analysis/data_test/data_GCDH_test_new/G_qc/'
save_folder = 'Figs_QC'
compound_set_input = ['B02', 'D02', 'F02']

plate_id_list = os.listdir(input_path)
plate_id_list

['.DS_Store', 'G-1041', 'G-1040', 'GG-251', 'G-1042']

In [104]:
for plate_id in plate_id_list:
    if len(plate_id.split('.')) > 1:
        continue

    working_dir = os.path.join(input_path, plate_id)
    # working_dir = f'/Users/behnam/Desktop/My_Codes/screen_analysis/data_test/data_GCDH_test_new/G_qc/{plate_id}/'
    
    # read the layout file and convert it into the well_positions_dict
    layout = pd.read_excel(os.path.join(working_dir,"TSA_Layout.xlsx"), index_col=0)
    well_positions_dict = layout2dict(layout, working_dir)
    
    # Read the raw data
    remove_header(os.path.join(working_dir,'TSA_RawData.csv'))
    
    try:    # type I data with header being removed
        rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData_removed_header.csv'), index_col=0, encoding='ISO-8859-1')
    except: # type II and IV data with ; as separator and extra rows to be removed
        try:
            rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData.csv'), encoding='ISO-8859-1', sep=';', index_col = 0)
            rawdata = rawdata.drop(['SampleID'], axis=1)
            pattern = '^[A-Z][0-9]{1,2}$'
            to_keep = [bool(re.findall(pattern, str(i))) for i in rawdata.index]
            rawdata = rawdata.loc[to_keep]
            if len(rawdata.iloc[0,0].split(',')) == 1:       # II
                rawdata = rawdata.apply(pd.to_numeric, errors='coerce')
            else:                                            # IV
                rawdata = rawdata.apply(lambda x: x.str.replace(',', '.').astype(float))
                rawdata.columns = rawdata.columns.str.replace(',', '.')
        except: # type III data with ; as separator and headers
            remove_header(os.path.join(working_dir,'TSA_RawData.csv'), sep=';')
            rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData_removed_header.csv'), index_col=0, encoding='ISO-8859-1', sep=';')
            rawdata = rawdata.apply(pd.to_numeric, errors='coerce')
    
    # Get all proteins, ligands and doses
    protein_set = list(well_positions_dict.keys())
    compound_set = list(well_positions_dict[protein_set[0]].keys())
    try:
        dosage_set = list(well_positions_dict[protein_set[0]][compound_set[0]].keys())
    except:
        dosage_set = list(well_positions_dict[protein_set[0]][compound_set[1]].keys())
    # protein_set, compound_set, dosage_set
    
    # Get max_replicate
    max_replicate = 0
    for protein in protein_set:
        for compound in compound_set:
            for dosage in dosage_set:
                # try:
                well_positions_target = well_positions_dict[protein][compound][dosage] if compound != compound_ctrl else well_positions_dict[protein][compound]   
                # except:
                # well_positions_target = []
                if len(well_positions_target) > max_replicate:
                    max_replicate = len(well_positions_target)
            
    colnames = (['protein', 'compound', 'dosage', 'locations'] +
                [f'tm1_{i}' for i in range(1, max_replicate+1)] + ['tm_mean', 'tm_std'] +
                [f'tm2_{i}' for i in range(1, max_replicate+1)] + ['tm_mean', 'tm_std'] +
                [f'r_squared_{i}' for i in range(1, max_replicate+1)] +
                # [f'peak_count_{i}' for i in range(1, max_replicate+1)] + ['peak_count_mean', 'peak_count_std'] +
                [f'fl25_{i}' for i in range(1, max_replicate+1)]
           )
    

    compound_set = compound_set if compound_set_input is None else compound_set_input
    
    fit_curve = False
    df = pd.DataFrame(columns=colnames)
    curve = pd.DataFrame()
    
    # Loop over protein
    for protein in tqdm(protein_set):
        if protein == protein_ctrl:
            continue
        # Loop over compound
        for compound in compound_set:
            # check for 0 dosage
            if compound == compound_ctrl:
                row, crv = process2(well_positions_dict, rawdata,
                                    protein, compound,
                                    save_dir = working_dir,
                                    fit_curve = fit_curve,
                                    max_replicate = max_replicate)
                df.loc[len(df)] = row
                curve = pd.concat([curve, crv], axis=1)
            else:
                # Loop over dosage
                for dosage in dosage_set:
                    row, crv = process2(well_positions_dict, rawdata, 
                                        protein, compound, dosage, 
                                        save_dir = working_dir,
                                        fit_curve = fit_curve,
                                        max_replicate = max_replicate)
                    df.loc[len(df)] = row
                    curve = pd.concat([curve, crv], axis=1)
    
    
    path_to_save = f'{save_folder}/{plate_id}'
    if not os.path.exists(path_to_save):
        os.makedirs(path_to_save)
    
    path = working_dir + '/fig_final/'
    well_list = os.listdir(path)
    well_list = [i for i in well_list if i.split('_')[1] in compound_set]
    
    path_list = [path + i for i in well_list]
    
    images = load_images(path_list)
    
    text = [f"{plate_id}"] * 3
    save_three_on_one_pdf(
        images, 
        text, 
        os.path.join(path_to_save, plate_id + '.pdf')
    )

100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.84it/s]
